# Lab 1 — Onset detection: what counts as an onset?

The onset-detection function (ODF) is the front end of much rhythm analysis. This lab begins with controlled signals so that we can see **why different detectors react differently**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d
from IPython.display import Audio, display

np.set_printoptions(precision=3, suppress=True)

def unbiased_autocorrelation(x, max_lag=None, normalize=True):
    x = np.asarray(x, dtype=float).reshape(-1)
    if max_lag is None:
        max_lag = len(x)-1
    max_lag = min(int(max_lag), len(x)-1)
    r = np.empty(max_lag+1)
    for tau in range(max_lag+1):
        a=x[:len(x)-tau]; b=x[tau:]
        r[tau] = np.dot(a,b)/max(1,len(a))
    if normalize and r[0] > 0:
        r = r/r[0]
    return r

def phase_autocorrelation(x, lags, unbiased=True):
    x=np.asarray(x,dtype=float).reshape(-1)
    lags=np.asarray(lags,dtype=int)
    P=np.zeros((len(lags), int(lags.max())))
    C=np.zeros_like(P)
    N=len(x)
    for ir,tau in enumerate(lags):
        for phi in range(tau):
            idx=np.arange(phi, N-tau, tau, dtype=int)
            if len(idx):
                P[ir,phi]=np.sum(x[idx]*x[idx+tau])
                C[ir,phi]=len(idx)
    if unbiased:
        return P/np.where(C==0,1,C), C
    return P, C

def frame_signal(x, frame_length, hop_length, center=True):
    x=np.asarray(x,float).reshape(-1)
    if center:
        p=frame_length//2
        x=np.pad(x,(p,p))
    starts=np.arange(0,max(1,len(x)-frame_length+1),hop_length,dtype=int)
    return np.stack([x[s:s+frame_length] for s in starts],axis=1), starts

def rhythmogram(x, frame_length=256, hop_length=32, lag_min=8, lag_max=96):
    frames, starts = frame_signal(x, frame_length, hop_length, center=True)
    lags=np.arange(lag_min,lag_max+1)
    R=np.zeros((len(lags),frames.shape[1]))
    w=np.hanning(frame_length)
    for j in range(frames.shape[1]):
        f=(frames[:,j]-frames[:,j].mean())*w
        ac=unbiased_autocorrelation(f, lag_max, normalize=False)
        R[:,j]=np.maximum(ac[lags],0)
        m=R[:,j].max()
        if m>0: R[:,j]/=m
    return R,lags,starts

def dp_tempo_path(R, lags, smoothness=30.0):
    R=np.asarray(R,float)
    eps=1e-10
    obs=-np.log(R+eps)
    nP,nT=R.shape
    cost=np.full((nP,nT),np.inf); back=np.zeros((nP,nT),dtype=int)
    cost[:,0]=obs[:,0]
    lp=np.log(np.asarray(lags,float))
    trans=smoothness*(lp[:,None]-lp[None,:])**2
    for t in range(1,nT):
        M=cost[:,t-1][None,:]+trans
        back[:,t]=np.argmin(M,axis=1)
        cost[:,t]=obs[:,t]+M[np.arange(nP),back[:,t]]
    path=np.zeros(nT,dtype=int); path[-1]=np.argmin(cost[:,-1])
    for t in range(nT-2,-1,-1): path[t]=back[path[t+1],t+1]
    return np.asarray(lags)[path]

def simple_novelty(x, fs, n_fft=1024, hop=256):
    f,t,Z=signal.stft(x,fs=fs,window='hann',nperseg=n_fft,noverlap=n_fft-hop,boundary=None,padded=False)
    mag=np.abs(Z)
    flux=np.maximum(mag[:,1:]-mag[:,:-1],0).sum(axis=0)
    flux=np.r_[0,flux]
    if flux.max()>0: flux/=flux.max()
    return flux,t

## 1. Four controlled events

We synthesize four events: an impulse-like transient, a gradual attack, a spectral change with nearly constant energy, and a noisy transient.

In [ ]:
fs=8000
T=2.0
t=np.arange(int(T*fs))/fs
x=np.zeros_like(t)
# impulse-like burst
idx=int(.25*fs); n=np.arange(int(.12*fs)); x[idx:idx+len(n)] += np.exp(-n/(.02*fs))*np.sin(2*np.pi*180*n/fs)
# gradual attack
idx=int(.65*fs); n=np.arange(int(.35*fs)); env=np.minimum(1,n/(.18*fs))*np.exp(-n/(.5*fs)); x[idx:idx+len(n)] += .7*env*np.sin(2*np.pi*260*n/fs)
# spectral change, similar energy
idx=int(1.15*fs); n=np.arange(int(.3*fs)); x[idx:idx+len(n)] += .5*np.sin(2*np.pi*700*n/fs)
# noise burst
idx=int(1.55*fs); n=np.arange(int(.12*fs)); rng=np.random.default_rng(1); x[idx:idx+len(n)] += .45*np.exp(-n/(.025*fs))*rng.standard_normal(len(n))
plt.figure(figsize=(12,3)); plt.plot(t,x); plt.xlabel('time [s]'); plt.title('Controlled test signal'); plt.show()
display(Audio(x,rate=fs))

## 2. Compare simple energy change and spectral flux

Energy-based novelty responds primarily to amplitude change. Spectral flux can also react strongly when spectral content changes even without a large amplitude jump.

In [ ]:
frame=256; hop=64
frames,_=frame_signal(x,frame,hop,center=False)
E=np.sum(frames**2,axis=0)
E=np.maximum(np.diff(np.r_[E[0],E]),0)
if E.max()>0: E/=E.max()
flux,tf=simple_novelty(x,fs,n_fft=256,hop=64)
tf2=np.arange(len(E))*hop/fs
plt.figure(figsize=(12,4)); plt.plot(tf2,E,label='positive energy difference'); plt.plot(tf,flux,label='spectral flux'); plt.legend(); plt.xlabel('time [s]'); plt.title('Two onset-evidence functions'); plt.show()

## 3. Experiment

Change the duration of the gradual attack, reduce the noise burst amplitude, or replace the spectral-change event with a frequency glide. Observe which detector still produces a strong peak.

**Question:** an ODF is already a representation. What information about the original waveform has been intentionally discarded?